# PlantSeg Multi-Seed Retraining -- Lane 2 (seed 7)

One of **6 parallel Kaggle notebooks** covering a 2-new-seed (3 total, including the original
SEED=42 checkpoints) retraining pass across all 8 classifiers and all 6 segmenters, for the
Array revision's variance/robustness response to R4/R5. Run all 6 lanes at the same time across
your 6 available sessions -- no lane depends on another.

**Full 6-lane schedule** (each lane's estimated hours use this model's ORIGINAL single-seed
train_time_sec from its `result_*.pkl` -- actual time will vary by GPU tier, but this is the
basis the packing below was built on):

| Lane | Seed | Segmenters | Classifiers | Est. hours |
|---|---|---|---|---|
| 1 | 123 | UNetPP_resnet50 | GatedFusionLight | ~9.85h |
| 2 | 7   | UNetPP_resnet50 | GatedFusionLight | ~9.85h |
| 3 | 123 | SegFormer_b2, FPN_resnet50, UNet_resnet34 | ResNet50, MobileNetV3Large, ViTBase16 | ~9.51h |
| 4 | 7   | SegFormer_b2, FPN_resnet50, UNet_resnet34 | ResNet50, MobileNetV3Large, ViTBase16 | ~9.51h |
| 5 | 123 | DeepLabV3Plus_efficientnet-b3, DeepLabV3Plus_resnet50 | ConvNeXtV2Tiny, EfficientNetV2S, EfficientNetB0, SwinTiny | ~9.41h |
| 6 | 7   | DeepLabV3Plus_efficientnet-b3, DeepLabV3Plus_resnet50 | ConvNeXtV2Tiny, EfficientNetV2S, EfficientNetB0, SwinTiny | ~9.41h |

**Why this split:** UNet++/ResNet50 alone took 9.19h for ONE seed in the original run (by far
the slowest of the 14 models), so it gets its own lane per seed with only the lightest classifier
(GatedFusionLight, 0.66h) alongside it. Everything else is packed to keep every lane's estimate
comfortably under Kaggle's 12h session limit (~2-2.5h of margin on every lane) rather than
optimizing for a perfectly even split.

**This notebook (Lane 2, seed 7) trains:** segmenters ['UNetPP_resnet50'] and classifiers
['GatedFusionLight']. Hyperparameters, architectures, losses and schedulers are all unchanged from the
original Stage 1/Stage 2 trainers -- only `SEED` and the output directory differ, so these
results are directly comparable to the existing SEED=42 checkpoints.

**Attach as Kaggle datasets:** the PlantSeg image+mask+annotation dataset (Metadata.csv +
images/ + annotations/ + annotation_train.json/annotation_val.json/annotation_test.json).
No pretrained checkpoints need attaching -- this notebook trains from ImageNet initialization,
same as the original runs.

**After it finishes:** collect this lane's output folder(s) (`/kaggle/working/s1_results_seed7/`
and/or `/kaggle/working/s2_results_seed7/`) and bring them back to this folder's
`revision_work/multiseed/` so they can be aggregated with the other 5 lanes into mean +/- std
tables for the response letter.


In [1]:
# ============ SEGMENTER SECTION: IMPORTS + SEED ============
!pip install -q segmentation-models-pytorch albumentations transformers
import os, json, math, time, random, warnings, pickle
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd
from PIL import Image, ImageDraw
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
warnings.filterwarnings("ignore")

SEED = 7   # <-- same new seed as the classifier section above (original checkpoints used SEED=42)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("Device:", device, "| smp", smp.__version__, "| SEED:", SEED)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 3.9 MB/s eta 0:00:00
Device: cuda | smp 0.5.0 | SEED: 7


In [2]:
# ============ SEGMENTER SECTION: CONFIG ============
DATA_ROOT = Path("/kaggle/input/datasets/alenmaruf/plantseg/plantseg")
if not (DATA_ROOT/"Metadata.csv").exists():
    for c in [Path("/kaggle/input/plantseg"), Path("plantseg"), Path(".")]:
        if (c/"Metadata.csv").exists(): DATA_ROOT=c; break
assert (DATA_ROOT/"Metadata.csv").exists(), f"Metadata.csv not under {DATA_ROOT} -- edit DATA_ROOT."
OUTDIR = Path(f"/kaggle/working/s2_results_seed7"); FIG = OUTDIR/"figures"; MASK = OUTDIR/"masks"
for d in [OUTDIR, FIG, MASK]: d.mkdir(parents=True, exist_ok=True)

SEG_MODELS_TO_TRAIN = ['UNetPP_resnet50']   # this lane's segmenter subset
SEG_TASK = "binary"; NUM_SEG = 2; IMG_SIZE = 512; NUM_EPOCHS = 40; PATIENCE = 8; LR = 6e-4; WD = 1e-4
print("Segmenters to train (seed 7):", SEG_MODELS_TO_TRAIN)


Segmenters to train (seed 7): ['UNetPP_resnet50']


In [3]:
# ============ SEGMENTER SECTION: INDEX + BINARY MASKS ============
meta = pd.read_csv(DATA_ROOT/"Metadata.csv", encoding="utf-8-sig")
meta["split"] = meta["Split"].map({"Training":"train","Validation":"val","Test":"test"})
meta["filepath"] = meta.apply(lambda r: DATA_ROOT/"images"/r["split"]/r["Name"], axis=1)
def load_anns(split):
    d = json.load(open(DATA_ROOT/f"annotation_{split}.json")); by = defaultdict(list)
    for a in d["annotations"]: by[a["image_id"]].append(a)
    fn = {}
    for im in d["images"]:
        polys = [seg for a in by.get(im["id"],[]) for seg in a["segmentation"] if len(seg)>=6]
        fn[im["file_name"]] = (im["width"], im["height"], polys)
    return fn
def rasterize(w,h,polys):
    m = Image.new("L",(w,h),0); dr = ImageDraw.Draw(m)
    for seg in polys: dr.polygon(list(zip(seg[0::2], seg[1::2])), fill=1)
    return m
recs = []
for split in ["train","val","test"]:
    fn = load_anns(split); od = MASK/split; od.mkdir(parents=True, exist_ok=True)
    for r in meta[meta.split==split].itertuples():
        w,h,polys = fn.get(r.Name, (None,None,[])); mp = od/(Path(r.Name).stem+".png")
        if not mp.exists():
            if w is None:
                with Image.open(r.filepath) as i0: w,h = i0.size
            rasterize(w,h,polys).save(mp)
        recs.append({"name": r.Name, "filepath": str(r.filepath), "mask": str(mp), "split": split,
                     "disease": r.Disease, "host": r.Plant})
df = pd.DataFrame(recs)
train_df = df[df.split=="train"].reset_index(drop=True)
val_df   = df[df.split=="val"].reset_index(drop=True)
test_df  = df[df.split=="test"].reset_index(drop=True)
print(f"train={len(train_df)} val={len(val_df)} test={len(test_df)} | binary masks ready")


train=5367 val=846 test=1561 | binary masks ready


In [4]:
# ============ SEGMENTER SECTION: TRANSFORMS / DATASET ============
MEAN = [0.485,0.456,0.406]; STD = [0.229,0.224,0.225]
train_tf = A.Compose([A.RandomResizedCrop(size=(IMG_SIZE,IMG_SIZE), scale=(0.6,1.0)), A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2), A.Rotate(limit=20, p=0.5), A.RandomBrightnessContrast(0.2,0.2,p=0.5), A.Normalize(MEAN,STD), ToTensorV2()])
eval_tf = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.Normalize(MEAN, STD), ToTensorV2()])
class SegDS(Dataset):
    def __init__(s, d, t): s.df = d.reset_index(drop=True); s.t = t
    def __len__(s): return len(s.df)
    def __getitem__(s, i):
        r = s.df.iloc[i]; img = np.array(Image.open(r.filepath).convert("RGB")); m = np.array(Image.open(r["mask"]))
        a = s.t(image=img, mask=m); return a["image"], a["mask"].long()
def loaders(bs):
    return (DataLoader(SegDS(train_df,train_tf), batch_size=bs, shuffle=True, num_workers=2, pin_memory=True, drop_last=True),
            DataLoader(SegDS(val_df,eval_tf), batch_size=bs, shuffle=False, num_workers=2, pin_memory=True),
            DataLoader(SegDS(test_df,eval_tf), batch_size=bs, shuffle=False, num_workers=2, pin_memory=True))


In [5]:
# ============ SEGMENTER SECTION: MODEL (unified full-res logits) ============
from transformers import SegformerForSemanticSegmentation
SMP_SPECS = {"UNet_resnet34": (smp.Unet, "resnet34"), "UNetPP_resnet50": (smp.UnetPlusPlus, "resnet50"),
    "DeepLabV3Plus_resnet50": (smp.DeepLabV3Plus, "resnet50"),
    "DeepLabV3Plus_efficientnet-b3": (smp.DeepLabV3Plus, "efficientnet-b3"),
    "FPN_resnet50": (smp.FPN, "resnet50")}
SEGFORMER_HF = {"SegFormer_b2": "nvidia/mit-b2"}
class SegNet(nn.Module):
    def __init__(s, name, nc):
        super().__init__()
        if name in SMP_SPECS:
            arch, enc = SMP_SPECS[name]; s.net = arch(encoder_name=enc, encoder_weights="imagenet", in_channels=3, classes=nc); s.kind="smp"
        else:
            s.net = SegformerForSemanticSegmentation.from_pretrained(SEGFORMER_HF[name], num_labels=nc, ignore_mismatched_sizes=True); s.kind="hf"
    def forward(s, x):
        if s.kind=="smp": return s.net(x)
        lg = s.net(pixel_values=x).logits; return F.interpolate(lg, size=x.shape[-2:], mode="bilinear", align_corners=False)
def bs_for(name): return 6 if "b2" in name else 8


In [6]:
# ============ SEGMENTER SECTION: METRICS / LOSS / TRAIN ============
class Conf:
    def __init__(s, n): s.n=n; s.cm=np.zeros((n,n), np.int64)
    def add(s, gt, pr):
        k = (gt>=0)&(gt<s.n); s.cm += np.bincount(s.n*gt[k].astype(int)+pr[k], minlength=s.n**2).reshape(s.n,s.n)
    def scores(s):
        cm = s.cm.astype(float); tp = np.diag(cm); iou = tp/(cm.sum(1)+cm.sum(0)-tp+1e-9); dice = 2*tp/(cm.sum(1)+cm.sum(0)+1e-9)
        pr = cm.sum(1)>0
        return {"mIoU": float(np.nanmean(iou[pr])), "mDice": float(np.nanmean(dice[pr])),
                "pixel_acc": float(tp.sum()/(cm.sum()+1e-9)), "fg_IoU": float(iou[1]), "fg_Dice": float(dice[1])}, iou
def dice_ce(lg, t, n):
    ce = F.cross_entropy(lg, t); p = F.softmax(lg,1); oh = F.one_hot(t,n).permute(0,3,1,2).float()
    d = 1-((2*(p*oh).sum((0,2,3))+1)/(p.sum((0,2,3))+oh.sum((0,2,3))+1)).mean(); return ce+d
def evaluate(model, loader, n):
    model.eval(); cf = Conf(n)
    with torch.no_grad():
        for x,y in loader:
            pr = model(x.to(device)).argmax(1).cpu().numpy(); cf.add(y.numpy().ravel(), pr.ravel())
    return cf
def train_one(name, n=NUM_SEG, epochs=NUM_EPOCHS, patience=PATIENCE):
    model = SegNet(name, n).to(device); bs = bs_for(name); tl, vl, te = loaders(bs)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=device.type=="cuda")
    hist = {"train_loss":[], "val_mIoU":[], "val_fgIoU":[]}; best=-1; best_state=None; bad=0
    for ep in range(epochs):
        model.train(); run=nb=0
        for x,y in tl:
            x,y = x.to(device), y.to(device); opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=device.type=="cuda"): loss = dice_ce(model(x), y, n)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); run += loss.item(); nb += 1
        sch.step(); sc, _ = evaluate(model, vl, n).scores()
        hist["train_loss"].append(run/max(nb,1)); hist["val_mIoU"].append(sc["mIoU"]); hist["val_fgIoU"].append(sc["fg_IoU"])
        print(f"[{name}] ep{ep+1}/{epochs} loss={run/max(nb,1):.3f} val_mIoU={sc['mIoU']:.4f} val_fgIoU={sc['fg_IoU']:.4f}")
        if sc["mIoU"] > best+1e-4: best=sc["mIoU"]; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}; bad=0
        else:
            bad += 1
            if bad >= patience: print(f"  early stop @ ep{ep+1}"); break
    model.load_state_dict(best_state); torch.save(best_state, OUTDIR/f"{name}_seed{SEED}.pt")
    return model, hist, bs
def deployment(model, img=IMG_SIZE, nw=3, it=15):
    p = sum(x.numel() for x in model.parameters()); tmp = OUTDIR/"_t.pt"; torch.save(model.state_dict(), tmp)
    sz = os.path.getsize(tmp)/1e6; os.remove(tmp); d = torch.randn(1,3,img,img)
    def t(dev):
        m = model.to(dev).eval(); dd = d.to(dev)
        with torch.no_grad():
            for _ in range(nw): m(dd)
            if dev.type=="cuda": torch.cuda.synchronize()
            t0 = time.time()
            for _ in range(it): m(dd)
            if dev.type=="cuda": torch.cuda.synchronize()
            return (time.time()-t0)/it*1000
    lg = t(device) if device.type=="cuda" else np.nan; lc = t(torch.device("cpu")); model.to(device)
    return {"params_M": p/1e6, "ckpt_MB": sz, "latency_cpu_ms": lc, "latency_gpu_ms": lg}
def per_image_fg_iou(model):
    model.eval(); v = []
    with torch.no_grad():
        for r in test_df.itertuples():
            img = np.array(Image.open(r.filepath).convert("RGB")); g = np.array(Image.open(r.mask))
            a = eval_tf(image=img, mask=g); pr = model(a["image"].unsqueeze(0).to(device)).argmax(1)[0].cpu().numpy()
            gg = a["mask"].numpy()>0; pp = pr>0; inter=(gg&pp).sum(); uni=(gg|pp).sum(); v.append(inter/uni if uni>0 else 1.0)
    return np.array(v)


In [7]:
# ============ SEGMENTER SECTION: MAIN LOOP (this lane's models, SEED 7) ============
for MODEL_NAME in SEG_MODELS_TO_TRAIN:
    print(f"\n{'='*60}\n  Training: {MODEL_NAME}  (seed 7)\n{'='*60}\n")
    t0 = time.time()
    model, hist, bs = train_one(MODEL_NAME)
    train_time = time.time()-t0
    cf = evaluate(model, loaders(bs)[2], NUM_SEG); sc, iou = cf.scores(); dep = deployment(model)
    pifg = per_image_fg_iou(model)
    res = {"model_name": MODEL_NAME, "seed": 7, "seg_task": SEG_TASK, "history": hist, "test_scores": sc,
           "iou": iou.tolist(), "deployment": dep, "per_image_fg_iou": pifg, "img_size": IMG_SIZE, "train_time_sec": train_time}
    pickle.dump(res, open(OUTDIR/f"s2_result_{MODEL_NAME}_seed7.pkl", "wb"))
    print("SAVED", OUTDIR/f"s2_result_{MODEL_NAME}_seed7.pkl")
    print("TEST:", {k: round(v,4) for k,v in sc.items()})
print("\nSegmenter section done for this lane.")



  Training: UNetPP_resnet50  (seed 7)



config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

[UNetPP_resnet50] ep1/40 loss=0.808 val_mIoU=0.6014 val_fgIoU=0.3759
[UNetPP_resnet50] ep2/40 loss=0.736 val_mIoU=0.6509 val_fgIoU=0.4609
[UNetPP_resnet50] ep3/40 loss=0.703 val_mIoU=0.6478 val_fgIoU=0.4558
[UNetPP_resnet50] ep4/40 loss=0.677 val_mIoU=0.6697 val_fgIoU=0.4894
[UNetPP_resnet50] ep5/40 loss=0.663 val_mIoU=0.6493 val_fgIoU=0.4419
[UNetPP_resnet50] ep6/40 loss=0.651 val_mIoU=0.6758 val_fgIoU=0.4947
[UNetPP_resnet50] ep7/40 loss=0.633 val_mIoU=0.6555 val_fgIoU=0.4537
[UNetPP_resnet50] ep8/40 loss=0.619 val_mIoU=0.6874 val_fgIoU=0.5179
[UNetPP_resnet50] ep9/40 loss=0.609 val_mIoU=0.6909 val_fgIoU=0.5188
[UNetPP_resnet50] ep10/40 loss=0.600 val_mIoU=0.6973 val_fgIoU=0.5288
[UNetPP_resnet50] ep11/40 loss=0.598 val_mIoU=0.6774 val_fgIoU=0.5176
[UNetPP_resnet50] ep12/40 loss=0.583 val_mIoU=0.7025 val_fgIoU=0.5452
[UNetPP_resnet50] ep13/40 loss=0.581 val_mIoU=0.7015 val_fgIoU=0.5360
[UNetPP_resnet50] ep14/40 loss=0.575 val_mIoU=0.6962 val_fgIoU=0.5356
[UNetPP_resnet50] ep15/40 los

In [8]:
# ============ CLASSIFIER SECTION: IMPORTS + SEED ============
!pip install -q timm
import os, json, math, time, random, warnings, pickle
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision; from torchvision import transforms
import timm
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, precision_score,
                             recall_score, matthews_corrcoef, confusion_matrix, roc_auc_score)
warnings.filterwarnings("ignore")

SEED = 7   # <-- new seed for this run (original paper checkpoints used SEED=42)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("Device:", device, "| SEED:", SEED)
assert device.type == "cuda", "No GPU detected -- set Settings > Accelerator (T4 x2 / P100) before Run All."


Device: cuda | SEED: 7


In [9]:
# ============ CLASSIFIER SECTION: CONFIG ============
# Unchanged from the original Stage 1 trainers except SEED (above), OUTDIR (seed-tagged so this
# run's checkpoints never overwrite the original SEED=42 ones), and MODELS_TO_TRAIN (this lane
# trains only its assigned subset -- see the intro markdown cell for the full 6-lane schedule).
DATA_ROOT = Path("/kaggle/input/plantseg/plantseg")
if not (DATA_ROOT/"Metadata.csv").exists():
    for cand in [Path("/kaggle/input/plantseg"), Path("plantseg"), Path(".")]:
        if (cand/"Metadata.csv").exists(): DATA_ROOT=cand; break
assert (DATA_ROOT/"Metadata.csv").exists(), f"Metadata.csv not found under {DATA_ROOT} -- edit DATA_ROOT."

OUTDIR = Path(f"/kaggle/working/s1_results_seed7"); FIG = OUTDIR/"figures"
OUTDIR.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)

MODELS_TO_TRAIN = ['GatedFusionLight']   # this lane's classifier subset

SCOPE = "full"
FRUIT_HOSTS = {"Apple","Citrus","Grape","Banana","Peach","Blueberry",
               "Plum","Cherry","Raspberry","Strawberry"}

NUM_EPOCHS      = 40
PATIENCE        = 8
LR              = 3e-4
WEIGHT_DECAY    = 1e-4
USE_PRETRAINED  = True
USE_CLASS_WEIGHTS = True
LABEL_SMOOTHING = 0.05
IMG_SIZE_CNN    = 384   # lower to 320/256 if OOM -- CNN classifiers trained at 384x384
USE_RANDAUG     = True
USE_TTA         = True
QUICK_SMOKE     = False

print("DATA_ROOT:", DATA_ROOT); print("Models to train (seed 7):", MODELS_TO_TRAIN)


AssertionError: Metadata.csv not found under /kaggle/input/plantseg/plantseg -- edit DATA_ROOT.

In [ ]:
# ============ CLASSIFIER SECTION: DATA INDEX ============
meta = pd.read_csv(DATA_ROOT/"Metadata.csv", encoding="utf-8-sig")
meta["split"] = meta["Split"].map({"Training":"train","Validation":"val","Test":"test"})
meta["filepath"] = meta.apply(lambda r: DATA_ROOT/"images"/r["split"]/r["Name"], axis=1)
meta["label"] = meta["Disease"]; meta["plant"] = meta["Plant"]
if SCOPE == "fruit":
    meta = meta[meta["plant"].isin(FRUIT_HOSTS)].reset_index(drop=True)
meta = meta[meta["filepath"].apply(lambda p: Path(p).exists())].reset_index(drop=True)

CLASS_NAMES = sorted(meta["label"].unique().tolist())
NUM_CLASSES = len(CLASS_NAMES)
label2idx = {c: i for i, c in enumerate(CLASS_NAMES)}

train_df = meta[meta.split=="train"].reset_index(drop=True)
val_df   = meta[meta.split=="val"].reset_index(drop=True)
test_df  = meta[meta.split=="test"].reset_index(drop=True)

counts = train_df["label"].map(label2idx).value_counts().reindex(range(NUM_CLASSES)).fillna(0)
inv = np.where(counts.values > 0, 1.0/counts.values, 0.0)
cw  = inv / inv[inv>0].mean()
class_weights = torch.tensor(cw, dtype=torch.float32, device=device)

print(f"classes={NUM_CLASSES}  train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")


In [ ]:
# ============ CLASSIFIER SECTION: TRANSFORMS / DATASET ============
MEAN = [0.485, 0.456, 0.406]; STD = [0.229, 0.224, 0.225]

def make_transforms(sz):
    ops = [transforms.RandomResizedCrop(sz, scale=(0.7,1.0)),
           transforms.RandomHorizontalFlip(),
           transforms.RandomVerticalFlip(p=0.2),
           transforms.RandAugment(num_ops=2, magnitude=9),
           transforms.ToTensor(), transforms.Normalize(MEAN, STD),
           transforms.RandomErasing(p=0.25)]
    ev = transforms.Compose([transforms.Resize((sz,sz)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
    return transforms.Compose(ops), ev

class DS(Dataset):
    def __init__(self, d, t):
        self.df = d.reset_index(drop=True); self.t = t
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        return self.t(Image.open(r.filepath).convert("RGB")), label2idx[r.label]


In [ ]:
# ============ CLASSIFIER SECTION: MODEL ZOO ============
# Verbatim from Stage 1's build_backbone() -- NOTE GatedFusionLight is intentionally NOT a case
# here: it is handled via FUSION_MODELS below and hardcoded to img_size=224 in train_one(),
# matching how it was actually trained (confirmed from source 2026-09-19; the paper originally
# implied it was 384 like the other CNNs -- also corrected in PlantSeg_after_review/paper.tex).
def build_backbone(name, num_classes, pretrained=USE_PRETRAINED):
    if name == "MobileNetV3Large":
        m = torchvision.models.mobilenet_v3_large(weights="IMAGENET1K_V2" if pretrained else None)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, num_classes)
        return m, IMG_SIZE_CNN, 24
    if name == "EfficientNetB0":
        return timm.create_model("efficientnet_b0", pretrained=pretrained, num_classes=num_classes), IMG_SIZE_CNN, 24
    if name == "ResNet50":
        m = torchvision.models.resnet50(weights="IMAGENET1K_V2" if pretrained else None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        return m, IMG_SIZE_CNN, 16
    if name == "EfficientNetV2S":
        return timm.create_model("tf_efficientnetv2_s", pretrained=pretrained, num_classes=num_classes), IMG_SIZE_CNN, 12
    if name == "ConvNeXtV2Tiny":
        return timm.create_model("convnextv2_tiny", pretrained=pretrained, num_classes=num_classes), IMG_SIZE_CNN, 16
    if name == "ViTBase16":
        return timm.create_model("vit_base_patch16_224", pretrained=pretrained, num_classes=num_classes), 224, 16
    if name == "SwinTiny":
        return timm.create_model("swin_tiny_patch4_window7_224", pretrained=pretrained, num_classes=num_classes), 224, 16
    raise ValueError(name)

class _Gated(nn.Module):
    def __init__(self, ba, da, bb, db, nc, gh, hh, dr):
        super().__init__(); self.branch_a=ba; self.branch_b=bb; fused=da+db
        self.gate = nn.Sequential(nn.Linear(fused,gh), nn.ReLU(), nn.Linear(gh,2), nn.Softmax(dim=1))
        self.classifier = nn.Sequential(nn.Linear(fused,hh), nn.ReLU(), nn.Dropout(dr), nn.Linear(hh,nc))
    def forward(self, x):
        fa, fb = self.branch_a(x), self.branch_b(x)
        g = self.gate(torch.cat([fa,fb], 1))
        return self.classifier(torch.cat([fa*g[:,0:1], fb*g[:,1:2]], 1))

def GatedFusionLightModel(nc, pretrained=USE_PRETRAINED):
    mb = torchvision.models.mobilenet_v3_large(weights="IMAGENET1K_V2" if pretrained else None)
    da = mb.classifier[0].in_features; mb.classifier = nn.Identity()
    b  = timm.create_model("efficientnet_b0", pretrained=pretrained, num_classes=0, global_pool="avg")
    return _Gated(mb, da, b, b.num_features, nc, 128, 256, 0.3)

FUSION_MODELS = {"GatedFusionLight": GatedFusionLightModel}


In [ ]:
# ============ CLASSIFIER SECTION: TRAIN / METRICS HELPERS ============
def train_one(model_name, num_epochs=NUM_EPOCHS, patience=PATIENCE):
    if model_name in FUSION_MODELS:
        model = FUSION_MODELS[model_name](NUM_CLASSES).to(device); img_size, bs = 224, 16
    else:
        model, img_size, bs = build_backbone(model_name, NUM_CLASSES); model = model.to(device)
    ttf, etf = make_transforms(img_size)
    tl = DataLoader(DS(train_df, ttf), batch_size=bs, shuffle=True,  num_workers=2, pin_memory=True, drop_last=True)
    vl = DataLoader(DS(val_df,   etf), batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)
    te = DataLoader(DS(test_df,  etf), batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)
    is_tf = model_name in ("ViTBase16","SwinTiny")
    lr = 5e-5 if is_tf else LR; warm_ep = 3 if is_tf else 2; pat = 10 if is_tf else patience
    opt  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    warm = torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.1, total_iters=warm_ep)
    cos  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, num_epochs-warm_ep))
    sch  = torch.optim.lr_scheduler.SequentialLR(opt, [warm, cos], milestones=[warm_ep])
    crit = nn.CrossEntropyLoss(weight=class_weights if USE_CLASS_WEIGHTS else None, label_smoothing=LABEL_SMOOTHING)
    scaler = torch.cuda.amp.GradScaler(enabled=device.type=="cuda")
    hist = {"train_loss":[], "val_loss":[], "train_acc":[], "val_acc":[]}; best=-1; best_state=None; bad=0
    for ep in range(num_epochs):
        model.train(); run=cor=tot=0
        for x,y in tl:
            x,y = x.to(device),y.to(device); opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=device.type=="cuda"):
                out = model(x); loss = crit(out, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            run += loss.item()*x.size(0); cor += (out.argmax(1)==y).sum().item(); tot += y.size(0)
        trl, tra = run/tot, cor/tot
        model.eval(); vr=vc=vt=0
        with torch.no_grad():
            for x,y in vl:
                x,y = x.to(device),y.to(device); out = model(x); loss = crit(out,y)
                vr += loss.item()*x.size(0); vc += (out.argmax(1)==y).sum().item(); vt += y.size(0)
        vlo, vac = vr/vt, vc/vt; sch.step()
        for k,v in zip(hist, [trl,vlo,tra,vac]): hist[k].append(v)
        print(f"[{model_name}] ep{ep+1}/{num_epochs} lr={opt.param_groups[0]['lr']:.1e} "
              f"trL={trl:.3f} vL={vlo:.3f} trA={tra:.3f} vA={vac:.3f}")
        if vac > best+1e-4:
            best=vac; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}; bad=0
        else:
            bad += 1
            if ep >= warm_ep and bad >= pat:
                print(f"  early stop @ ep{ep+1}"); break
    model.load_state_dict(best_state); torch.save(best_state, OUTDIR/f"{model_name}_best.pt")
    model.eval(); P=[]; Y=[]; PR=[]
    with torch.no_grad():
        for x,y in te:
            x = x.to(device); lg = model(x)
            if USE_TTA: lg = lg + model(torch.flip(x, dims=[3]))
            p = F.softmax(lg, 1).cpu().numpy(); P.extend(p.argmax(1)); Y.extend(y.numpy()); PR.extend(p)
    return model, img_size, hist, np.array(Y), np.array(P), np.array(PR)

def deployment(model, img_size, nw=5, ni=30):
    n = sum(p.numel() for p in model.parameters())
    tmp = OUTDIR/"_t.pt"; torch.save(model.state_dict(), tmp)
    sz = os.path.getsize(tmp)/1e6; os.remove(tmp)
    d  = torch.randn(1, 3, img_size, img_size)
    def t(dev):
        m = model.to(dev).eval(); dd = d.to(dev)
        with torch.no_grad():
            for _ in range(nw): m(dd)
            if dev.type=="cuda": torch.cuda.synchronize()
            t0 = time.time()
            for _ in range(ni): m(dd)
            if dev.type=="cuda": torch.cuda.synchronize()
            return (time.time()-t0)/ni*1000
    lg = t(device) if device.type=="cuda" else float("nan")
    lc = t(torch.device("cpu")); model.to(device)
    return {"params_M": n/1e6, "checkpoint_size_MB": sz, "latency_cpu_ms": lc, "latency_gpu_ms": lg}

def compute_metrics(y, yp, ypb, n):
    cm = confusion_matrix(y, yp, labels=range(n)); specs=[]
    for i in range(n):
        tn = cm.sum()-cm[i,:].sum()-cm[:,i].sum()+cm[i,i]; fp = cm[:,i].sum()-cm[i,i]
        specs.append(tn/(tn+fp) if (tn+fp)>0 else float("nan"))
    present = [c for c in range(n) if (y==c).sum()>0]; aucs=[]
    for c in present:
        try: aucs.append(roc_auc_score((y==c).astype(int), ypb[:,c]))
        except Exception: pass
    return {"accuracy": accuracy_score(y,yp),
            "balanced_accuracy": balanced_accuracy_score(y,yp),
            "f1_macro": f1_score(y,yp,average="macro"),
            "f1_weighted": f1_score(y,yp,average="weighted"),
            "precision_macro": precision_score(y,yp,average="macro",zero_division=0),
            "recall_macro": recall_score(y,yp,average="macro",zero_division=0),
            "mcc": matthews_corrcoef(y,yp),
            "roc_auc_macro": float(np.mean(aucs)) if aucs else float("nan")}, cm


In [ ]:
# ============ CLASSIFIER SECTION: MAIN LOOP (this lane's models, SEED 7) ============
for MODEL_NAME in MODELS_TO_TRAIN:
    print(f"\n{'='*60}\n  Training: {MODEL_NAME}  (seed 7)\n{'='*60}\n")
    t0 = time.time()
    model, img_size, hist, Y, P, PR = train_one(MODEL_NAME)
    train_time = time.time()-t0
    metrics, cm = compute_metrics(Y, P, PR, NUM_CLASSES)
    dep = deployment(model, img_size)
    result = {"model_name": MODEL_NAME, "seed": 7, "history": hist,
              "y_true": Y, "y_pred": P, "y_prob": PR,
              "metrics": metrics, "cm": cm, "deployment": dep,
              "img_size": img_size, "train_time_sec": train_time, "class_names": CLASS_NAMES}
    pickle.dump(result, open(OUTDIR/f"result_{MODEL_NAME}_seed7.pkl", "wb"))
    print("SAVED", OUTDIR/f"result_{MODEL_NAME}_seed7.pkl")
    print({k: round(v,4) for k,v in metrics.items() if isinstance(v, float)})
print("\nClassifier section done for this lane.")


## Lane 2 (seed 7) complete

All assigned models trained and saved. Bring the output pkl/checkpoint files back for aggregation.